In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

import xgboost
from xgboost import XGBClassifier
from xgboost import to_graphviz

In [ ]:
def classification_metrics(y_actual, y_pred):
    acc = round(accuracy_score(y_actual, y_pred)*100, 2)
    precision = round(precision_score(y_actual, y_pred)*100, 2)
    recall = round(recall_score(y_actual, y_pred)*100, 2)
    f1 = round(f1_score(y_actual, y_pred)*100, 2)
    return {"Accuracy":acc, "Precision":precision, "Recall":recall, "F1":f1}

In [ ]:
# Loading dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
columns = [
    'age','workclass','fnlwgt','education','education_num','marital_status',
    'occupation','relationship','race','sex','capital_gain','capital_loss',
    'hours_per_week','native_country','income'
]

In [ ]:
df = pd.read_csv(url, header=None, names=columns, na_values=" ?")
df.head()

In [ ]:
df.income.value_counts()

In [ ]:
df.shape

In [ ]:
df.isna().sum()

In [ ]:
df = df.dropna()
df.isna().sum()

In [ ]:
df.shape

In [ ]:
# encoding categorical variables
le = LabelEncoder()
for col in df.select_dtypes(include="object").columns:
    df[col] = le.fit_transform(df[col])

In [ ]:
X = df.drop("income", axis=1)
y = df["income"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# decision tree
dt = DecisionTreeClassifier(max_depth=3)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)
classification_metrics(y_test, y_pred_dt)

In [ ]:
plt.figure()
plt.bar(X.columns, dt.feature_importances_)
plt.xticks(rotation=90)
plt.title("DT Feature Importance")
plt.show()

In [ ]:
plt.figure(figsize=(16,10))
plot_tree(dt, feature_names=X.columns, filled=True)
plt.title("Decision Tree")
plt.show()

In [ ]:
# random forest
rf = RandomForestClassifier(n_estimators=100, max_depth=10)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
classification_metrics(y_test, y_pred_rf)

In [ ]:
plt.figure()
plt.bar(X.columns, rf.feature_importances_)
plt.xticks(rotation=90)
plt.title("RF Feature Importance")
plt.show()

In [ ]:
# gradient boosting
gb = GradientBoostingClassifier(n_estimators=100)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)
classification_metrics(y_test, y_pred_gb)

In [ ]:
plt.figure()
plt.bar(X.columns, gb.feature_importances_)
plt.xticks(rotation=90)
plt.title("GB Feature Importance")
plt.show()

In [ ]:
# visualize one tree
plt.figure(figsize=(16,10))
plot_tree(gb.estimators_[0,0], feature_names=X.columns, filled=True)
plt.title("GB First Tree")
plt.show()

In [ ]:
# XGBoost
xgb = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, eval_metric="logloss")
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)
classification_metrics(y_test, y_pred_xgb)

In [ ]:
# Feature importance
plt.figure()
plt.bar(X.columns, xgb.feature_importances_)
plt.xticks(rotation=90)
plt.title("XGBoost Feature Importance")
plt.show()

In [ ]:
plt.figure(figsize=(40, 30))
xgboost.plot_tree(xgb, tree_idx=0) # plots the first tree at index 0
plt.show()

In [ ]:
# saving to a file
dot = to_graphviz(xgb, tree_idx=0)
dot.render("xgboost_tree")

In [ ]:
# best-performing model
best_pred = y_pred_xgb
cm = confusion_matrix(y_test, best_pred)
plt.figure()
sns.heatmap(cm, annot=True, fmt="d")
plt.title("Confusion Matrix (XGBoost)")
plt.show()

**Exercise**
1. Use cross-validation on the above problem. How does this affect performance if at all?


In [ ]:
# Part 1 Using cross-validation  on XG Boost

from sklearn.model_selection import cross_val_score

# Train with cross-validation , k= 5
xgb = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    eval_metric="logloss"
)
cv_scores = cross_val_score(
    xgb,
    X_train,
    y_train,
    cv=5,
    scoring='accuracy'
)
print("Cross-validation scores:", cv_scores)
print("Mean CV accuracy:", cv_scores.mean())

# Train normally with cross validation
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)
classification_metrics(y_test, y_pred_xgb)

The performance of the XGBoost model did not change significantly after applying cross-validation. The cross-validation accuracy (0.8694) was very close to the accuracy obtained using a single train-test split (0.8644). This indicates that the model is stable and generalizes well to unseen data. Cross-validation mainly provided a more reliable estimate of performance rather than improving it.

2. Select two models and perform hyperparameter optimization to select the best hyperparameters. How does this affect the performance of the models?

In [ ]:
# Part 2
# Hyperparamter optimization for Random forests

from sklearn.model_selection import GridSearchCV

params = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2,5]
}

rf = RandomForestClassifier()
grid_rf = GridSearchCV(rf, params, cv=5, scoring='accuracy')
grid_rf.fit(X_train, y_train)

best_rf = grid_rf.best_estimator_

print(grid_rf.best_params_)
print(grid_rf.best_score_)

In [ ]:
#Hyperparameter optimization for Gradient Boosting

gb = GradientBoostingClassifier()

param_grid_gb = {
    'learning_rate': [0.01, 0.1],
    'n_estimators': [100, 200],
    'max_depth': [3, 5]
}

grid_gb = GridSearchCV(
    gb,
    param_grid_gb,
    cv=5,
    scoring='accuracy'
)

grid_gb.fit(X_train, y_train)

best_gb = grid_gb.best_estimator_

print(grid_gb.best_params_)
print(grid_gb.best_score_)

The performance improvement was minimal, that is to say Random forests without optimizing hyperparameter(85.28) and with hyperparamter optimization (0.8587) and for Gradient boosting before is (85.89) and after (0.8702).
Indicating that the default parameters were already close to optimal for this dataset.


3. Choose a simple regression problem of your choice and use the above algorithms for prediction. Provide a comparison of their performance and their feature importance.

In [ ]:
#Part 3
# Regression problem for predicting house prices

# using house price prediction dataset
df = pd.read_csv(r"C:\Users\user\Desktop\Data sets\House Price Prediction Dataset.csv")

# Encoding features
# Garage presence or absence
df['Garage'] = df['Garage'].map({'Yes': 1, 'No': 0})

# Condition of House
condition_map = {
    'Poor': 0,
    'Fair': 1,
    'Good': 2,
    'Excellent': 3
}

df['Condition'] = df['Condition'].map(condition_map)

# Location of house
df = pd.get_dummies(df, columns=['Location'], drop_first=True)

# Feature matrix and target vector
features = df.drop(columns = ["Price","Id"])
target = df["Price"]
X = features
y = target

# Train Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
#Train the 4 models
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

models = {
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest": RandomForestRegressor(),
    "Gradient Boosting": GradientBoostingRegressor(),
    "XGBoost": XGBRegressor()
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    score = model.score(X_test, y_test)  # R² score
    results[name] = score
    print(f"{name}: {score}")

In [ ]:
# Visualize Performance

names = list(results.keys())
scores = list(results.values())

plt.figure()
plt.bar(names, scores)
plt.title("Model Perfomance Comparison (R² Score)")
plt.xlabel("Models")
plt.ylabel("R² Score")
plt.xticks(rotation=20)
plt.show()

# Extract Feature Importance

feature_importance = {}

for name, model in models.items():
    if hasattr(model, "feature_importances_"):
        feature_importance[name] = model.feature_importances_

# Virtualize feature Importance
feature_names = X_train.columns  

for name, importance in feature_importance.items():
    plt.figure()
    plt.barh(feature_names, importance)
    plt.title(f"{name} Feature Importance")
    plt.xlabel("Importance")
    plt.ylabel("Features")
    plt.show()